# 02 — Analysis

Loads experiment data from Google Drive and runs all analyses.

**Run `01_run_experiment.ipynb` first** to generate `data_MODEL_DATE.csv` files.

## 1. Setup

In [ ]:
# Clone repo (if not already done in this session)
!git clone https://github.com/auertobias/authority-bias-llm.git 2>/dev/null || echo "Already cloned"
%cd authority-bias-llm

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q statsmodels

import os, sys, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf

sys.path.insert(0, '.')
from src.config import DATA_PATH, RESULTS_PATH

os.makedirs(RESULTS_PATH, exist_ok=True)
print(f"Reading data from: {DATA_PATH}")
print(f"Saving results to: {RESULTS_PATH}")

## 2. Load Data
Automatically finds and combines all `data_MODEL_DATE.csv` files.

In [ ]:
data_files = sorted(glob.glob(DATA_PATH + "data_*.csv"))

if not data_files:
    raise FileNotFoundError(
        f"No data files found in {DATA_PATH}\n"
        "Run 01_run_experiment.ipynb first."
    )

print(f"Found {len(data_files)} file(s):")
dfs = []
for f in data_files:
    tmp = pd.read_csv(f)
    dfs.append(tmp)
    print(f"  {os.path.basename(f):40s} {len(tmp)} rows")

df_raw = pd.concat(dfs, ignore_index=True)
df = df_raw.dropna(subset=['rating']).copy()
df['rating'] = df['rating'].astype(int)

print(f"\nCombined: {len(df_raw)} total, {len(df)} valid ratings ({100*len(df)/len(df_raw):.1f}%)")
print(f"Models:   {df['model'].unique().tolist()}")

## 3. Quality Checks

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Rating distribution
df['rating'].hist(bins=20, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Rating Distribution (1–100)')
axes[0].set_xlabel('Rating')
axes[0].axvline(df['rating'].mean(), color='red', ls='--',
                label=f"Mean: {df['rating'].mean():.1f}")
axes[0].legend()

# Sequential independence
for name, mdf in df.groupby('model'):
    mdf = mdf.sort_index().reset_index(drop=True)
    axes[1].scatter(range(len(mdf)), mdf['rating'], alpha=0.15, s=5, label=name)
axes[1].set_title('Ratings by Trial Order')
axes[1].set_xlabel('Trial order')
axes[1].set_ylabel('Rating')
axes[1].legend()

# Per-model boxplot
df.boxplot(column='rating', by='model', ax=axes[2])
axes[2].set_title('Rating by Model')
axes[2].set_xlabel('')
plt.suptitle('')

plt.tight_layout()
plt.savefig(RESULTS_PATH + 'quality_checks.png', dpi=150, bbox_inches='tight')
plt.show()

print("Sequential correlation by model:")
for name, mdf in df.groupby('model'):
    mdf = mdf.sort_index().reset_index(drop=True)
    r, p = stats.pearsonr(range(len(mdf)), mdf['rating'])
    flag = " ⚠️" if p < 0.05 else " ✓"
    print(f"  {name:10s} r={r:.3f}, p={p:.3f}{flag}")

## 4. Descriptive Statistics

In [ ]:
STATUS_ORDER = ['low', 'medium', 'high']

print("═══ Mean Rating by Status ═══")
print(df.groupby('status')['rating'].agg(['mean','std','count']).reindex(STATUS_ORDER).round(2))

print("\n═══ Mean Rating by Expertise ═══")
print(df.groupby('expertise')['rating'].agg(['mean','std','count']).round(2))

print("\n═══ Mean Rating by Argument Type ═══")
print(df.groupby('arg_type')['rating'].agg(['mean','std','count']).round(2))

print("\n═══ Status × Expertise ═══")
print(df.groupby(['status','expertise'])['rating'].mean().unstack().reindex(STATUS_ORDER).round(2))

print("\n═══ Status × Argument Type ═══")
print(df.groupby(['status','arg_type'])['rating'].mean().unstack().reindex(STATUS_ORDER).round(2))

In [ ]:
# Per-model breakdown (if multiple models)
if df['model'].nunique() > 1:
    print("═══ Per-Model: Mean Rating by Status ═══")
    for name, mdf in df.groupby('model'):
        print(f"\n--- {name.upper()} ---")
        print(mdf.groupby('status')['rating'].agg(['mean','std','count']).reindex(STATUS_ORDER).round(2))

## 5. Regressions

In [ ]:
# Model 1: Main effects
m1 = smf.ols(
    'rating ~ C(status, Treatment(reference="low")) + C(expertise) + C(arg_type)',
    data=df
).fit(cov_type='cluster', cov_kwds={'groups': df['trial_id']})

print("═══ MODEL 1: Main Effects ═══")
print(m1.summary())

In [ ]:
# Model 2: Status × Argument Type interaction
m2 = smf.ols(
    'rating ~ C(status, Treatment(reference="low")) * C(arg_type) + C(expertise)',
    data=df
).fit(cov_type='cluster', cov_kwds={'groups': df['trial_id']})

print("═══ MODEL 2: Status × Argument Type ═══")
print(m2.summary())

In [ ]:
# Model 3: Full three-way interaction
m3 = smf.ols(
    'rating ~ C(status, Treatment(reference="low")) * C(expertise) * C(arg_type)',
    data=df
).fit(cov_type='cluster', cov_kwds={'groups': df['trial_id']})

print("═══ MODEL 3: Full Three-Way ═══")
print(m3.summary())

In [ ]:
# Model 4: With model fixed effects (auto-activates with multiple models)
if df['model'].nunique() > 1:
    m4 = smf.ols(
        'rating ~ C(status, Treatment(reference="low")) + C(expertise) + C(arg_type) + C(model)',
        data=df
    ).fit(cov_type='cluster', cov_kwds={'groups': df['trial_id']})
    print("═══ MODEL 4: Main Effects + Model FE ═══")
    print(m4.summary())
else:
    print("Single model — skipping model fixed effects.")

## 6. Effect Sizes

In [ ]:
status_eff    = df[df['status']=='high']['rating'].mean() - df[df['status']=='low']['rating'].mean()
expertise_eff = df[df['expertise']=='relevant']['rating'].mean() - df[df['expertise']=='irrelevant']['rating'].mean()
type_eff      = df[df['arg_type']=='normative']['rating'].mean() - df[df['arg_type']=='descriptive']['rating'].mean()

print(f"Status    (high − low):              {status_eff:+.2f}")
print(f"Expertise (relevant − irrelevant):   {expertise_eff:+.2f}")
print(f"Type      (normative − descriptive): {type_eff:+.2f}")

if abs(status_eff) > abs(expertise_eff):
    ratio = abs(status_eff / expertise_eff) if expertise_eff != 0 else float('inf')
    print(f"\n→ Status effect is {ratio:.1f}× larger → STATUS DOMINATES")
else:
    ratio = abs(expertise_eff / status_eff) if status_eff != 0 else float('inf')
    print(f"\n→ Expertise effect is {ratio:.1f}× larger → EXPERTISE DOMINATES")

# Cohen's d
high = df[df['status']=='high']['rating']
low  = df[df['status']=='low']['rating']
pooled_sd = np.sqrt((high.std()**2 + low.std()**2) / 2)
d = (high.mean() - low.mean()) / pooled_sd
print(f"\nCohen's d (status high vs low): {d:.3f}")

## 7. The Killer Figure

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
STATUS_ORDER = ['low', 'medium', 'high']

# Panel A: Status × Expertise
means_a = df.groupby(['status','expertise'])['rating'].mean().unstack().reindex(STATUS_ORDER)
sems_a  = df.groupby(['status','expertise'])['rating'].sem().unstack().reindex(STATUS_ORDER)
means_a.plot(kind='bar', ax=axes[0], color=['#2196F3','#FF9800'],
             width=0.7, yerr=sems_a, capsize=3)
axes[0].set_title('Rating by Status & Domain Expertise', fontsize=12)
axes[0].set_xlabel('Speaker Status')
axes[0].set_ylabel('Mean Rating (1–100)')
axes[0].legend(title='Expertise', loc='lower right')
axes[0].set_xticklabels(['Low','Medium','High'], rotation=0)

# Panel B: Status × Argument Type
means_b = df.groupby(['status','arg_type'])['rating'].mean().unstack().reindex(STATUS_ORDER)
sems_b  = df.groupby(['status','arg_type'])['rating'].sem().unstack().reindex(STATUS_ORDER)
means_b.plot(kind='bar', ax=axes[1], color=['#4CAF50','#9C27B0'],
             width=0.7, yerr=sems_b, capsize=3)
axes[1].set_title('Rating by Status & Argument Type', fontsize=12)
axes[1].set_xlabel('Speaker Status')
axes[1].legend(title='Arg Type', loc='lower right')
axes[1].set_xticklabels(['Low','Medium','High'], rotation=0)

plt.tight_layout()
plt.savefig(RESULTS_PATH + 'authority_bias_figure.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved to {RESULTS_PATH}authority_bias_figure.png")

## 8. Weakness Length Analysis

In [ ]:
df['weakness_length'] = df['weaknesses'].fillna('').str.len()

print("Mean weakness length by status:")
print(df.groupby('status')['weakness_length'].mean().reindex(STATUS_ORDER).round(1))

m_wl = smf.ols(
    'weakness_length ~ C(status, Treatment(reference="low")) + C(expertise) + C(arg_type)',
    data=df
).fit(cov_type='cluster', cov_kwds={'groups': df['trial_id']})
print("\n═══ Weakness Length Regression ═══")
print(m_wl.summary())

## 9. Export

In [ ]:
summary = df.groupby(['status','expertise','arg_type'])['rating'].agg(['mean','std','count']).round(2)
summary.to_csv(RESULTS_PATH + 'summary_table.csv')

coefs = pd.DataFrame({
    'M1_coef': m1.params.round(3), 'M1_pval': m1.pvalues.round(4),
    'M2_coef': m2.params.round(3), 'M2_pval': m2.pvalues.round(4),
})
coefs.to_csv(RESULTS_PATH + 'regression_coefficients.csv')

print(f"Saved to {RESULTS_PATH}:")
print("  summary_table.csv")
print("  regression_coefficients.csv")
print("  authority_bias_figure.png")
print("  quality_checks.png")
display(summary)